# CS4765/6765 Assignment 2: Sentiment Analysis for climate-related text in corporate disclosures

**Due 14 October**

We've seen the problem of sentiment analysis in class. In this
assignment you will build your own sentiment analysis system for a three-way sentiment analysis task for climate-related text in corporate disclosures. You will consider logistic regression using several sets of features, including the example features from the lecture and the textbook, count-based and tf-idf-based unigram features, and features of your choosing. You will compare these approaches to each other and a most-frequent class baseline.

Read through this notebook in its entirety before getting started. You should only make changes / write code in parts of the notebook where the instructions ask you to do so. These are indicated with TODO throughout.

Your code must be able to run on the NLP VM on the lab machines. You may only import additional modules where the instructions indicate that you may do so.


## Data

I've provided you with the following files for this assignment: `train-sample.csv`, `dev.csv`, `test.csv`.

Each of these files is a CSV file. You will use `train-sample.csv` for training models, `dev.csv` for evaluation during model development and for model refinement, and `test.csv` for final evaluation.

Each row in each file is an instance consisting of a text, and a label. (Each row also contains an id number, which we won't use in this assignment.) Each text is a climate-related paragraph from an annual report from a company. Each label is 0, 1 or 2. The labels have the following meanings:

- 2: Opportunity arising due to climate change (positive sentiment)

- 1: Neutral

- 0: Risk or threat that negatively impacts an entity (negative sentiment)

`train-sample.csv`, `dev.csv`, and `test.csv` contain 800, 200, and 300 instances, respectively.

The starter code handles reading these files for you. Do not modify these parts of starter code. You should only modify the starter code where you are instructed to (indicated with TODO).

The data for this assignment is from the [ClimateBERT climate sentiment dataset](https://huggingface.co/datasets/climatebert/climate_sentiment). The license is [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) You are of course welcome to read more about this dataset, but it is not necessary to do so to be able to do this assignment. To create the data for this assignment, I converted the original dataset files from parquet to CSV. The original dataset does not include a development set. I randomly split the training data into 80% training and 20% dev to create `train-sample.csv` and `dev.csv`. `test.csv` is the same test data as the original dataset (but formatted as CSV instead of parquet).





## Experimental Setup

Throughout this assignment, we will always use the training data (`train-sample.csv`) for training models. We will implement our models and then use the development data (`dev.csv`) for preliminary evaluation. Once we've completed this, we will do our final evaluation on the test data (`test.csv`). The starter code guides you through this process.


In [1]:
# Helper function to load a data file
import csv

def get_texts_and_labels(fname):
    csv_reader = csv.reader(open(fname, encoding='utf8'))
    # Ignore header row
    next(csv_reader)
    texts = []
    labels = []
    for line in csv_reader:
        _,text,label = line
        label = int(label)
        texts.append(text)
        labels.append(label)
    return texts,labels

In [2]:
# Load the training and dev data
train_texts, train_labels = get_texts_and_labels('data/train-sample.csv')
dev_texts, dev_labels = get_texts_and_labels('data/dev.csv')

# Some sanity checks
assert len(train_texts) == len(train_labels)
assert len(dev_texts) == len(dev_labels)

for label in train_labels + dev_labels:
    assert 0 <= label <= 2

In [3]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# A helper function to compute macro averaged P, R, and F1 and accuracy.
# Uses implementantions of evaluation metrics from sklearn.
def get_results(gold_labels, predicted_labels):
    p,r,f,_ = precision_recall_fscore_support(gold_labels, 
                                              predicted_labels,
                                              average='macro',
                                              zero_division=0)
    acc = accuracy_score(gold_labels, predicted_labels)
    return p,r,f,acc    

# A helper function to print out the above evaluation metrics.
def print_results(gold_labels, predicted_labels):
    p,r,f,acc = get_results(gold_labels, predicted_labels)
    
    print("Precision: ", p)
    print("Recall: ", r)
    print("F1: ", f)
    print("Accuracy: ", acc)
    print()

## Most-frequent Class Baseline

The starter code below trains a most-frequent class baseline on the training data and evaluates it on the dev data. (Note that sklearn also includes an implementation of a most-frequent class baseline, which you might find useful for your projects: https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html)

In [4]:
# A most-frequent class baseline
class Baseline:
    def __init__(self, labels):
        self.train(labels)

    def train(self, labels):
        # Count classes to determine which is the most frequent
        label_freqs = {}
        for l in labels:
            label_freqs[l] = label_freqs.get(l, 0) + 1
        self.mfc = sorted(label_freqs, reverse=True, 
                          key=lambda x : label_freqs[x])[0]
    
    def predict(self, test_instance):
        # Ignore the test instance and always return the most frequent class
        return self.mfc


In [5]:
baseline_classifier = Baseline(train_labels)
baseline_dev_predictions = [baseline_classifier.predict(x) for x in dev_texts]
print_results(dev_labels, baseline_dev_predictions)

Precision:  0.14166666666666666
Recall:  0.3333333333333333
F1:  0.19883040935672514
Accuracy:  0.425



## Example Features from Textbook

The code below implements the example logistic regression features from lecture and the textbook (i.e., Chapter 4, Figure 4.2).

In [6]:
import re

# A very simple tokenizer. This implements sklearn's default
# word tokenization, which applies case folding and treats
# alphanumeric sequences of 2 or more characters as tokens.
# Punctuation is treated as a token separator, i.e.,
# punctuation never occurs in the resulting tokens.
word_tokenize_pattern = re.compile(r"(?u)\b\w\w+\b")
def word_tokenize(s, apply_case_folding=True):
    return [x.lower() for x in word_tokenize_pattern.findall(s)]


In [7]:
# Here is an example document from the training data and the result of tokenizing it
print(train_texts[58])
print(word_tokenize(train_texts[58]))

Target: Consume 100% renewable electricity by July 2021 in Europe and by 2025 in all markets
['target', 'consume', '100', 'renewable', 'electricity', 'by', 'july', '2021', 'in', 'europe', 'and', 'by', '2025', 'in', 'all', 'markets']


In [8]:
import math, nltk
import numpy as np

# First and second person pronouns based on information from 
# https://en.wikipedia.org/wiki/Personal_pronouns_in_English
first_second_pronouns = set(['i', 'me', 'my', 'mine', 'myself',
                             'we', 'us', 'our', 'ours', 'ourselves',
                             'you', 'your', 'yours', 'yourself', 'yourselves'])

# Lists of positive and negative sentiment words from the opinion lexicon included in NLTK
# https://www.nltk.org/_modules/nltk/corpus/reader/opinion_lexicon.html
# (If you've already downloaded this data you can comment the following line out.)
nltk.download('opinion_lexicon')
pos_words = set(nltk.corpus.opinion_lexicon.positive())
neg_words = set(nltk.corpus.opinion_lexicon.negative())

# Return a feature vector for the document doc
# The features correspond to the example from lecture and the textbook
# (i.e., Chapter 4, Figure 4.2)
def slp_features_for_doc(doc):
    tokens = word_tokenize(doc)
    x1 = len([t for t in tokens if t in pos_words])
    x2 = len([t for t in tokens if t in neg_words])
    x3 = 1 if "no" in tokens else 0
    x4 = 1 if "!" in doc else 0
    x5 = int(any([t in first_second_pronouns for t in tokens]))
    x6 = math.log(len(tokens))
    return np.array([x1, x2, x3, x4, x5, x6])


[nltk_data] Downloading package opinion_lexicon to
[nltk_data]     /Users/umairbutt/nltk_data...
[nltk_data]   Package opinion_lexicon is already up-to-date!


In [9]:
# Here is an example document from the training data and its feature vector
print(train_texts[425])
print(slp_features_for_doc(train_texts[425]))

Of these scenarios, 'Autonomy' is the scenario we consider best represents the technology and policy context that would be essential to meet the aspiration of limiting cumulative emissions to 450 ppm.
[2.        0.        0.        0.        1.        3.4339872]


The code below uses scikit-learn (sklearn) to train a logistic regression classifier with the example features on the training data and evaluate it on the dev data.

Scikit-learn is a popular tool for doing many machine learning tasks in Python. It includes implementations of many classifiers (including logistic regression). Read the comments in the code below to learn how to use it.

In [10]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# slp_train_features is a Dx6 matrix, where D is the number of training documents
# and 6 is the number of features
slp_train_features = np.vstack([slp_features_for_doc(x) for x in train_texts])

# Train a logistic regression classifier on the training
# data. A wide range of options are available. sklearn uses
# L2 regularization by default. The algorithm for minimizing
# the loss during training is LBFGS instead of SGD, which we
# saw in lecture. The random_state is set to 0  (an 
# arbitrarily chosen number) to help ensure results are 
# consistent from run to run.
# http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
lr_slp_classifier = LogisticRegression(random_state=0).fit(slp_train_features, train_labels)

# slp_dev_features is a Dx6 matrix, where here D is the number of
# dev documents
slp_dev_features = np.vstack([slp_features_for_doc(x) for x in dev_texts])

# Evaluate the classifier on the dev data
lr_slp_dev_predictions = lr_slp_classifier.predict(slp_dev_features)
print_results(dev_labels, lr_slp_dev_predictions)


Precision:  0.6033452807646356
Recall:  0.5441676421812506
F1:  0.5466931792290645
Accuracy:  0.595



## Part 1: Extended features (2 marks)

In this part of the assignment you will consider extending the example features from the textbook with features of your own choosing. Based on the discussion of document classification, logistic regression, and sentiment analysis in class, or analysis of the training and development data, identify some features (e.g., 1-5 or so) that you believe could potentially give an improvement when combined with the example features. For this part of the assignment, focus on features targeting specific properties of a document, such as for the example features (as opposed to more-general features, such as counts for all unigrams in a document, which we will consider later in the assignment).

(I've included some examples of analysis of the training and dev data below that could be helpful. You are of course welcome to carry out further analyses, although you should not look at the test data yet.) 

Some ideas of features to consider could include the presence or count of specific words, punctuation marks, or sets of characters, that could be indicative of the class. However, you are by no means limited to these ideas.

Answer the questions below. Implement your new features below that. Then run the code below that to train a logistic regression classifier using these features on the training data and evaluate it on the dev data. This classifier should improve over the classifier using just the example features.


Answer the following questions:

1. Which features are you adding? Provide a brief explanation.

In addition to already implemented six features, I have selected four more features, and their details are provided below.

    i. Count of words that convey the risk factor of climate change. This includes words like `risk`, `damage`, `loss`, `adverse`, and `disruption`.
    
    ii. Count of words that convey the meaning of opportunity. This includes `opportunity`, `benefit`, `growth`, `renewable`, and `efficiency`.
    
    iii. Count of words that convey the meaning of possibility. This includes `may`, `could`, `might`, and `would`.
    
    iv. Check the occurrence of bigram risk phrases like `adverse impact`, `adverse effect`, or `physical risk`.

I also tested adding a binary feature for the presence of % for various climate-related figures. But it reduced development macro F1 from 0.6633 to 0.6416, so I excluded it.

I also tested a temperature-word count. But it also reduced development macro F1 from 0.6633 to 0.6580. Then, I also tested temperature phrases like `extreme heat`, `rising temperatures`, and `heat stress`. But the resultant increase was very minute 0.004 (0.6677). So, I did not add this.

2. Why might we reasonably expect that a logistic regression classifier using these features and the example features would improve over a logistic regression classifier using just the example features?

Two example features count general positive and negative sentiment words from `opinion_lexicon`. The other features capture negation, punctuation, pronouns, and text length.  The features I selected are domain-specific and may provide additional information about climate risks and opportunities to the logistic regression classifier than the generic words. For example, the first set of words, associated with the risk factor of climate change, helps the model in identifying the negative impacts of climate change.

Similarly, the second set of features helps the model in identifying the potential business opportunities. The third set of features contains conditional words that describe possible or uncertain outcomes. It associates with events that could affect the company's operations.

It is important to highlight that these features are selected purely based on heuristics. They are not selected using any systemactic technique.

In [11]:
def new_features_for_doc(doc):
    #new_features is a list containing the new features.
    new_features = []
    tokens = word_tokenize(doc)

    risk_words = {"risk", "risks", "damage", "damages", "loss", "losses",
                  "adverse", "disruption", "disruptions"}
    opportunity_words = {"opportunity", "opportunities", "benefit", "benefits",
                         "growth", "renewable", "renewables", "efficiency"}
    possibility_words = {"may", "could", "might", "would"}

    new_features.append(len([t for t in tokens if t in risk_words]))
    new_features.append(len([t for t in tokens if t in opportunity_words]))
    new_features.append(len([t for t in tokens if t in possibility_words]))

    # bigram phrases for risk aspect
    bigram_risk_phrases = {"adverse impact", "adverse impacts",
                           "adverse effect", "adverse effects",
                           "physical risk", "physical risks"}
    risk_phrase_flag = 0
    for i in range(len(tokens) - 1):
        phrase = tokens[i] + " " + tokens[i + 1]
        if phrase in bigram_risk_phrases:
            risk_phrase_flag = 1
            break
    new_features.append(risk_phrase_flag)

    # Don't modify below here
    return np.hstack([slp_features_for_doc(doc), np.array(new_features)])

In [12]:
new_train_features = np.vstack([new_features_for_doc(x) for x in train_texts])
# max_iter is the maximum number of iterations during training. It is increased from the 
# default of 100 here to allow training to converge.
# If you encounter a convergence warning you could try increasing this further.
lr_new_classifier = LogisticRegression(random_state=0, max_iter=500).fit(new_train_features, train_labels)
new_dev_features = np.vstack([new_features_for_doc(x) for x in dev_texts])

lr_new_dev_predictions = lr_new_classifier.predict(new_dev_features)
print_results(dev_labels, lr_new_dev_predictions)


Precision:  0.7193787693787694
Recall:  0.6491817871427178
F1:  0.6632851809771141
Accuracy:  0.69



In [13]:
# Here are some example analyses of the behaviour of the logistic regression
# classifier with the example features that could be helpful for identifying 
# interesting features to consider.
#
# Compute a confusion matrix for logistic regression with the example
# features on the dev data. 
# Note that I've transposed the confusion matrix so that 
# the rows correspond to system predictions and columns correspond to
# gold standard labels, following the convention in the textbook 
# and class. (By default, sklearn does it the other way around.)
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html

from sklearn.metrics import confusion_matrix
print(confusion_matrix(dev_labels, lr_slp_dev_predictions, labels=[0, 1, 2]).transpose())

[[42 10  2]
 [23 66 35]
 [ 2  9 11]]


In [14]:
# Calculte per-class P, R, F for logistic regression with the example
# features on the dev data.
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html

for label in list(range(3)):
    print(label)
    # This prints P, R, F, and support (i.e., number of gold standard instances) for each class
    print(precision_recall_fscore_support(dev_labels, lr_slp_dev_predictions, labels=[label]))

0
(array([0.77777778]), array([0.62686567]), array([0.69421488]), array([67]))
1
(array([0.53225806]), array([0.77647059]), array([0.63157895]), array([85]))
2
(array([0.5]), array([0.22916667]), array([0.31428571]), array([48]))


In [15]:
# Sort the features by their weight for each class for
# the logistic regression classifier using the example features.
# Note that this is a 3-way classification problem. As such, the classifier
# here is multinomial logistic regression, which is slightly different
# than the (binary) logistic regression classifier we saw in lecture. In
# multinomial logistic regression, each class has a separate vector of weights.
# Note that feature numbers below start at 0, i.e., 0 corresponds to x1 in
# the example in the textbook.
for class_label_i,coefs in enumerate(lr_slp_classifier.coef_):
    print("Top features for class", lr_slp_classifier.classes_[class_label_i])
    for fi,val in sorted(enumerate(coefs), key=lambda x : x[1], reverse=True):
        print(fi, val)


Top features for class 0
1 0.6522591822718863
3 0.0
4 -0.09088636398931109
0 -0.1992916888180517
2 -0.5882822182768289
5 -0.6159368869658871
Top features for class 1
2 0.847179268885522
4 0.12382786167697918
0 0.03816742517786864
3 0.0
5 -0.11618201314406829
1 -0.11984128399776592
Top features for class 2
5 0.7321189001099729
0 0.16112426364017704
3 0.0
4 -0.03294149768766789
2 -0.25889705060869395
1 -0.5324178982741004


## Unigram Count Features

The code below uses sklearn to train a logistic regression classifier with unigram count features on the training data and evaluate it on the dev data. (This document representation is also commonly referred to as a "bag-of-words".)

In [16]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression

# sklearn provides functionality for tokenizing text and
# extracting features from it. This uses sklearn's default
# tokenization which applies case folding and treats
# alphanumeric sequences of 2 or more characters as tokens.
# Punctuation is treated as a token separator, i.e.,
# punctuation never occurs in the resulting tokens.
# http://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html
# (The word_tokenize function provided above is intended to
# produce the same tokenization.)
count_vectorizer = CountVectorizer()

# train_counts will be a DxV matrix where D is the number of
# training documents and V is the number of word (token) types 
# in the training documents. Each cell in the matrix indicates 
# the frequency (count) of a word type in a document.
train_counts = count_vectorizer.fit_transform(train_texts)

# Train a logistic regression classifier on the training
# data. A wide range of options are available. sklearn uses
# L2 regularization by default. The algorithm for minimizing
# the loss during training is LBFGS instead of SGD, which we
# saw in lecture. The random_state is set to 0  (an 
# arbitrarily chosen number) to help ensure results are 
# consistent from run to run.
# http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
lr = LogisticRegression(random_state=0)
lr_classifier = lr.fit(train_counts, train_labels)

# Transform the dev documents into a DxV matrix, similar to
# that for the training documents, where D is the number of
# dev documents, and V is the number of word types in the 
# training documents.
dev_counts = count_vectorizer.transform(dev_texts)

# Predict the class for each dev document. 
lr_dev_predictions = lr_classifier.predict(dev_counts)

print_results(dev_labels, lr_dev_predictions)

Precision:  0.7900114973711068
Recall:  0.7734038142620232
F1:  0.7801965694519533
Accuracy:  0.78



## Part 2: Tf-idf (2 marks)

We will compare logistic regression with unigram count features to the same approach but using tf-idf weighting instead of raw counts. I.e., here we will consider logistic regression with tf-idf weighted unigram features.

Using the code above for unigram count features as an example, and documentation below on the sklearn TfidfVectorizer, add code below to implement this tf-idf approach. As in the above example, your classifier should be trained on the training data and evaluated on the dev data. In implementing this approach, you should aim to make your classifier the same as the above example, but with one difference: the features should use tf-idf weighting instead of simple counts.

https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

You should not import any additional modules. 

Hint: This should not be a huge amount of code. (My sample solution is less than 10 lines of code.) If you are writing a lot of code you might be off track.


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
lr_tfidf_classifier = LogisticRegression(random_state=0).fit(train_tfidf, train_labels)
dev_tfidf = tfidf_vectorizer.transform(dev_texts)
lr_tfidf_dev_predictions = lr_tfidf_classifier.predict(dev_tfidf)
print_results(dev_labels, lr_tfidf_dev_predictions)

Precision:  0.8240139734054542
Recall:  0.7850136572041753
F1:  0.7987237076648842
Accuracy:  0.8



## Part 3: Other Features (4 marks)

In this part of the assignment you will again consider logistic regression with features of your choosing. Based on the discussion of document classification, logistic regression, and sentiment analysis in class, or analysis of the training and development data, identify a document representation and corresponding features that you believe could potentially improve over tf-idf weighted unigram features.

(Examining a confusion matrix or the per-class P, R, and F1 values for the classifier using tf-idf weighted unigram features could be helpful. The analysis code from Part 1 provides examples that you could adapt for this. I have included an additional example below for examining feature weights in the classifier using unigram count features that could also be helpful. You are of course welcome to carry out further analyses, although you should not look at the test data yet.)

Implement a logistic regression classifier using these features, train it on the training data, and evaluate it on the dev data.

Here are some ideas for features, but you are by no means limited to them!

* word $n$-grams (perhaps using tf-idf weighting)
* character $n$-grams (again perhaps using tf-idf weighting)
* tf-idf weighted unigram features, but incorporating case folding (or not), stop-word removal, stemming, etc.
* tf-idf weighted unigram features, but using alternative tokenization methods (perhaps incorporating information that isn't present in the default tokenization, such as punctuation)

Importantly, although you need to be able to explain why the features you consider could potentially improve over tf-idf weighted unigram features, it is not a requirement for this assignment that these features actually do improve over tf-idf weighted unigram features. (Many approaches that you might reasonably expect to give an improvement turn out not to for this dataset. For this part of the assignment, the motivation for considering the features and discussion of findings is more important than getting a high F1 or accuracy.)

For implementing your features, note that the constructor for sklearn's TfidfVectorizer has many optional parameters. For example, ```ngram_range``` could be helpful for some of the suggested features above. If you would like to explore alternative approaches to tokenization, ```tokenizer``` allows you to pass your own tokenizer.
https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

For this part of the assignment, you are welcome to import other modules from within sklearn, nltk, numpy, or scipy; however, you must use the sklearn logistic regression classifier.

In the approach you consider, it might be necessary to set some hyper-parameters. For example, if you consider word $n$-gram features, it might be necessary to set the value of $n$. You could do this by grid search over a reasonable range of values, training on the training data and evaluating on the dev data. If you explore the use of normalization approaches, such as case folding and stop-word removal, you could explore combinations of these on the development data to identify the best approach. (Importantly, you should not consider the test data at this point.)


Answer the following questions:

1. Which features are you using? Provide a brief explanation.

I am using TF-IDF to extract single words (unigrams) and word combinations (n-grams). I will test two settings:

    - Ungrams and bigrams that I will extract using `ngram_range` parameter and will set its value to (1, 2).

    - Unigram, bigrams, and trigrams that I will extract using the same `ngram_range` parameter. But its value will be (1, 3).

Whichever setting gives the better development macro F1 score will be selected for final testing.
   
1. Why might we reasonably expect that a logistic regression classifier using these features would potentially improve over a logistic regression classifier using tf-idf weighted unigram features?

Unigrams look at single words and ignore word order. But bigrams and trigrams capture short phrases like `extreme weather` and `renewable energy`. This can help the classifier better understand the meaning of the text and identify different categories.

However, longer phrases may appear less often in the training data. Since there are only 800 training examples, they may not always improve classification performance. The development data is used to check whether these features improve the model or not.


In [18]:
# Compare two small choices using development macro F1.
best_dev_f1 = 0
for max_n in [2, 3]:
    vectorizer = TfidfVectorizer(ngram_range=(1, max_n))
    train_features = vectorizer.fit_transform(train_texts)
    classifier = LogisticRegression(random_state=0).fit(train_features, train_labels)
    dev_features = vectorizer.transform(dev_texts)
    predictions = classifier.predict(dev_features)
    p, r, f1, acc = get_results(dev_labels, predictions)
    print("Maximum n:", max_n, "Dev F1:", f1, "Dev accuracy:", acc)

    if f1 > best_dev_f1:
        best_dev_f1 = f1
        best_max_n = max_n
        other_vectorizer = vectorizer
        lr_other_classifier = classifier
        lr_other_dev_predictions = predictions

print("Selected maximum n:", best_max_n)
print_results(dev_labels, lr_other_dev_predictions)


Maximum n: 2 Dev F1: 0.7804272040460397 Dev accuracy: 0.785
Maximum n: 3 Dev F1: 0.7911993690710654 Dev accuracy: 0.795
Selected maximum n: 3
Precision:  0.8305221800904564
Recall:  0.7730770168764023
F1:  0.7911993690710654
Accuracy:  0.795



In [19]:
# Find the top-10 words (features) with highest weight for each class for
# the logistic regression classifier using unigram count features.
# It could be helpful to adapt this to your classifier using
# tf-idf weighted unigram features and your tfidf Vectorizer.
for class_label_i,coefs in enumerate(lr_classifier.coef_):
    print("Top-10 words for class", lr_classifier.classes_[class_label_i])
    top_n = sorted(enumerate(coefs), key=lambda x : x[1], reverse=True)[:10]
    for vocab_i,weight in top_n:
        print (count_vectorizer.get_feature_names_out()[vocab_i], weight)
    print()


Top-10 words for class 0
risks 0.7970069773398878
risk 0.7297130776430021
potential 0.592064357928893
may 0.5720899257170838
weather 0.5169030893730489
physical 0.4404389504249364
impact 0.440379649049727
events 0.40000648555108426
or 0.3919893028041971
impacts 0.3909574534200499

Top-10 words for class 1
target 0.4947676245607261
governance 0.46527119560846725
based 0.451323813918469
emissions 0.4296994395852789
thermal 0.41003389015780356
scope 0.3920788829301676
committee 0.3916850132506049
no 0.3432868533958687
analysis 0.3425437581139652
scenarios 0.3277207924798728

Top-10 words for class 2
million 0.773680381235924
billion 0.6344998439700581
sustainable 0.6256614808636511
efficiency 0.6183780718547055
solutions 0.5484273506080859
service 0.48598586176646463
committed 0.4673218197524402
fund 0.4602642174463066
opportunities 0.4426440891584198
green 0.4165022518539305



## Part 4: Test Data and Report (2 marks)

So far, we've only evaluated on the development data. Once you have completed the tasks above, run the code below to evaluate the classifiers on the test data.

In [20]:
test_texts,test_labels = get_texts_and_labels('data/test.csv')

print("Test results:")
print()

print("Baseline:")
baseline_test_predictions = [baseline_classifier.predict(x) for x in test_texts]
print_results(test_labels, baseline_test_predictions)

print("SLP Example Features")
slp_test_features = np.vstack([slp_features_for_doc(x) for x in test_texts])
slp_test_predictions = lr_slp_classifier.predict(slp_test_features)
print_results(test_labels, slp_test_predictions)

print("SLP Example + Extended Features")
new_test_features = np.vstack([new_features_for_doc(x) for x in test_texts])
new_test_predictions = lr_new_classifier.predict(new_test_features)
print_results(test_labels, new_test_predictions)

print("Unigram Count Features")
test_counts = count_vectorizer.transform(test_texts)
lr_test_predictions = lr_classifier.predict(test_counts)
print_results(test_labels, lr_test_predictions)

print("Tf-idf Weighted Unigram Features")
# run your logistic regression classifier using 
# tf-idf weighted unigram features on the test data
test_tfidf = tfidf_vectorizer.transform(test_texts)
lr_tfidf_test_predictions = lr_tfidf_classifier.predict(test_tfidf)
print_results(test_labels, lr_tfidf_test_predictions)

print("Part 3: Other Features")
# run your logistic regression classifier using 
# features of your choice from Part 3 on the test data
other_test_features = other_vectorizer.transform(test_texts)
lr_other_test_predictions = lr_other_classifier.predict(other_test_features)
print_results(test_labels, lr_other_test_predictions)

Test results:

Baseline:
Precision:  0.16979166666666667
Recall:  0.3333333333333333
F1:  0.22498274672187715
Accuracy:  0.509375

SLP Example Features
Precision:  0.5415126737008739
Recall:  0.5362802218545325
F1:  0.538683175017988
Accuracy:  0.59375

SLP Example + Extended Features
Precision:  0.651771310805529
Recall:  0.6372575499312663
F1:  0.6424092714008681
Accuracy:  0.68125

Unigram Count Features
Precision:  0.6405949791868577
Recall:  0.6391179383355766
F1:  0.6397733310398869
Accuracy:  0.684375

Tf-idf Weighted Unigram Features
Precision:  0.6965208564045774
Recall:  0.6674088549646041
F1:  0.6766528005338697
Accuracy:  0.73125

Part 3: Other Features
Precision:  0.7341996233521657
Recall:  0.6692582732054894
F1:  0.6807404540763673
Accuracy:  0.753125




In this assignment you have considered several logistic regression classifiers. Write a brief report describing your findings that addresses at least the following points. In addressing these points, focus on results on the test data, but you are welcome to also comment on results on dev data where it is relevant to your claims.

1. Do the features you added in Part 1 improve over the example features. I.e., does logistic regression using the example features and the features you added in Part 1 improve over logistic regression using just the example features?

1. Does tf-idf weighting improve performance? I.e., does logistic regression with tf-idf weighted unigram features improve over logistic regression with unigram count features?

1. Does logistic regression using the features of your choice (from Part 3) improve over logistic regression using tf-idf weighted unigram features? Provide some explanation for why this might be the case.

For example, if logistic regression using the features of your choice outperforms logistic regression using tf-idf weighted unigram features, you could carry out some error analysis to attempt to explain what causes the difference. For this, you might find it helpful to examine the per-class P, R, and F values, or a confusion matrix. (Examples of how to do this are given above on the dev data, but here you would want to consider the test data.)

On the other hand, if logistic regression using the features of your choice does not outperform logistic regression using tf-idf weighted unigram features, attempt to explain why this might be the case.


1. Do part 1 features improve performance?

Yes. On the test data, adding the risk-word count, opportunity-word count, possibility-word count, and risk-phrase flag increased macro F1 from 0.5387 to 0.6424 and accuracy from 0.5938 to 0.6813. These corpus-specific features provide relevant information to the model about climate risks, opportunities, and possible outcomes that the general sentiment features may miss.

2. Does TF-IDF weighting improve performance?

Yes. Compared with unigram counts, TF-IDF increased test macro F1 from 0.6398 to 0.6767 and accuracy from 0.6844 to 0.7313. TF-IDF gives relatively more weight to words that occur in fewer documents. Its normalization also reduces differences caused by paragraph length.

3. Do part 3 features improve performance?

Yes, the selected combination of unigrams, bigrams, and trigrams in Part 3, slightly improved the model’s performance compared to TF-IDF unigrams. Macro F1 increased from 0.6767 to 0.6807, and accuracy increased from 0.7313 to 0.7531. The model performed better for risk and neutral classes but worse for the opportunity class. Short phrases helped the model understand the text better, but many phrases appeared rarely in the 800 training examples. Because of this, the overall improvement was small.